# Embedded System Experiments
## Lab: Systolic Arrays vs. Pipelined Adder Trees

This notebook explores the hardware architectures used to accelerate General Matrix Multiplication (**GEMM**) and General Matrix-Vector Multiplication (**GEMV**).

### Learning Objectives:
1.  Understand the data flow patterns in a **Systolic Array**.
2.  Visualize the logarithmic reduction of an **Adder Tree**.
3.  Compare the arithmetic intensity and throughput of **GEMM** vs. **GEMV**.

## 1. The Role of Buffers and Staggering
In hardware, data doesn't appear everywhere at once.
* **Input Buffers:** Store rows of Matrix A and columns of Matrix B.
* **Staggering:** To ensure that $A_{0,1}$ meets $B_{1,0}$ at the correct Processing Element (PE) at the correct time, we must "skew" the input. The second row starts 1 cycle after the first, the third row 2 cycles after, and so on.


## 2. GEMM and Systolic Arrays
A **Systolic Array** is a network of PEs that pass data to their neighbors.
* **Formula:** $C = A \times B$
* **Mechanism:** In this simulation, $A$ moves **Right**, $B$ moves **Down**, and the result $C$ is accumulated locally in each PE (**Output Stationary**).

## 3. GEMV and Adder Trees
An **Adder Tree** is used when we need to sum many products quickly (a Dot Product).
* **Formula:** $y = A \times x$
* **Mechanism:** Multipliers compute all $A_{i,j} \times x_j$ in parallel. Then, a pipelined tree of adders reduces these values to a single sum in $O(\log_2 N)$ cycles.

In [ ]:
import numpy as np
import time
from IPython.display import clear_output, display

class SystolicArraySimulator:
    def __init__(self, size):
        self.size = size
        # Internal state of PEs: {'acc': accumulator, 'a': current_a, 'b': current_b}
        self.pe_grid = [[{'acc': 0, 'a': 0, 'b': 0} for _ in range(size)] for _ in range(size)]

    def _format_input_buffers(self, A, B):
        """
        Skew logic:
        A[r, k] enters row r at cycle r + k
        B[k, c] enters col c at cycle c + k
        """
        # Total cycles needed for all data to pass through: ~3 * size
        total_time = 3 * self.size

        # a_stream[row][time], b_stream[col][time]
        a_stream = np.full((self.size, total_time), None, dtype=object)
        b_stream = np.full((self.size, total_time), None, dtype=object)

        for r in range(self.size):
            for c in range(self.size):
                # A streams in from the left: row 'r' at time 'r + c'
                a_stream[r, r + c] = A[r, c]
                # B streams in from the top: col 'c' at time 'c + r'
                b_stream[c, c + r] = B[r, c]

        return a_stream, b_stream

    def _format_horizontal_matrices(self, A, B, Res):
        """Helper to string-ify matrices and place them side-by-side."""
        a_lines = str(A).split('\n')
        b_lines = str(B).split('\n')
        r_lines = str(Res).split('\n')

        # Determine height for padding (in case of weird formatting)
        max_h = max(len(a_lines), len(b_lines), len(r_lines))

        # Pad lines to equal height
        a_lines += [""] * (max_h - len(a_lines))
        b_lines += [""] * (max_h - len(b_lines))
        r_lines += [""] * (max_h - len(r_lines))

        output = []
        output.append(f"{'Matrix A':<20}   {'Matrix B':<20}   {'Expected (A@B)':<20}")
        output.append("-" * 80)

        for i in range(max_h):
            row = f"{a_lines[i]:<20}   {b_lines[i]:<20} = {r_lines[i]:<20}"
            output.append(row)

        return "\n".join(output)

    def run(self, A, B):
        a_stream, b_stream = self._format_input_buffers(A, B)
        total_cycles = a_stream.shape[1]
        expected = np.matmul(A, B)

        # Generate the static side-by-side string once
        matrix_header = self._format_horizontal_matrices(A, B, expected)

        # --- STATIC DISPLAY SECTION ---
        # This part stays at the top
        static_header = f"""
{'='*80}
SYSTOLIC ARRAY GEMM SIMULATION: {self.size}x{self.size}
{'='*80}
{matrix_header}

{'='*80}
CYCLE LEVEL SIMULATION
{'='*80}
"""

        for t in range(total_cycles):
            # 1. Generate the animation frame
            display_str = self._generate_display(t, a_stream, b_stream)

            # 2. Refresh the whole output but rewrite static info + new frame
            clear_output(wait=True)
            print(static_header)
            print(display_str)

            # 3. Logic: Update Accumulators
            for r in range(self.size):
                for c in range(self.size):
                    pe = self.pe_grid[r][c]
                    if pe['a'] is not None and pe['b'] is not None:
                        pe['acc'] += pe['a'] * pe['b']

            # 4. Shift Data (Movement)
            # Iterate backwards to avoid overwriting data in the same cycle
            for r in range(self.size - 1, -1, -1):
                for c in range(self.size - 1, -1, -1):
                    # Update A (Horizontal flow)
                    if c == 0:
                        self.pe_grid[r][c]['a'] = a_stream[r, t] if t < a_stream.shape[1] else None
                    else:
                        self.pe_grid[r][c]['a'] = self.pe_grid[r][c-1]['a']

                    # Update B (Vertical flow)
                    if r == 0:
                        self.pe_grid[r][c]['b'] = b_stream[c, t] if t < b_stream.shape[1] else None
                    else:
                        self.pe_grid[r][c]['b'] = self.pe_grid[r-1][c]['b']

            time.sleep(2)

    def _generate_display(self, t, a_stream, b_stream):
        lines = []
        lines.append(f"Cycle: {t}")
        lines.append("-" * 80)

        vis_depth = self.size
        left_margin_width = (vis_depth * 4) + 5
        margin_str = " " * left_margin_width

        # TOP SECTION (B Streaming)
        for d in range(vis_depth, 0, -1):
            row_str = margin_str
            for c in range(self.size):
                idx = t + d - 1
                val = b_stream[c, idx] if idx < b_stream.shape[1] else None
                val_str = f"{val:>3}" if val is not None else "  ."
                row_str += f"  {val_str}   "
            lines.append(row_str)

        lines.append(margin_str + "    ￬   " * self.size)

        # MIDDLE SECTION (A Streaming + PEs)
        for r in range(self.size):
            left_str = ""
            for d in range(vis_depth, 0, -1):
                idx = t + d - 1
                val = a_stream[r, idx] if idx < a_stream.shape[1] else None
                val_str = f"{val:>3}" if val is not None else "  ."
                left_str += f"{val_str} "

            left_str += " -> "
            line_top = left_str
            line_bot = " " * len(left_str)

            for c in range(self.size):
                pe = self.pe_grid[r][c]
                a_v = pe['a'] if pe['a'] is not None else 0
                b_v = pe['b'] if pe['b'] is not None else 0
                acc_v = pe['acc']

                line_top += f"[{a_v:>2}|{b_v:>2}] "
                line_bot += f" Σ {acc_v:<3} "

            lines.append(line_top)
            lines.append(line_bot)
            lines.append("")

        return "\n".join(lines)


class AdderTreeSimulator:
    def __init__(self, size):
        self.size = size
        self.stages = int(np.log2(size))

        # Pipeline State:
        # stage 0: Multiplier outputs (size N)
        # stage 1..S: Adder outputs (size N/2, N/4...)
        # stored as dictionaries {'val': value, 'row_id': row_index} to track data flow
        self.pipeline = []
        current_w = size
        for _ in range(self.stages + 1):
            self.pipeline.append([None] * current_w)
            current_w //= 2

        self.results = [] # Stores final calculated results
        self.weights = None # Will hold Vector V

    def _format_horizontal_matrices(self, A, V, Res):
        """Helper to display A, V, and Expected Result side-by-side statically."""
        a_lines = str(A).split('\n')
        v_lines = str(V).split('\n')
        r_lines = str(Res).split('\n')

        max_h = max(len(a_lines), len(v_lines), len(r_lines))

        # Pad with empty lines
        a_lines += [""] * (max_h - len(a_lines))
        v_lines += [""] * (max_h - len(v_lines))
        r_lines += [""] * (max_h - len(r_lines))

        output = []
        output.append(f"{'Matrix A':<25}   {'Vector V':<15}   {'Expected Result':<15}")
        output.append("-" * 80)

        for i in range(max_h):
            # Formatted columns
            row = f"{a_lines[i]:<25} * {v_lines[i]:<15} = {r_lines[i]:<15}"
            output.append(row)

        return "\n".join(output)

    def _generate_tree_display(self, t, n_rows):
        lines = []

        # --- CONSTANTS FOR LAYOUT ---
        BLOCK_WIDTH = 9   # Width of [aa|bb]
        GAP_WIDTH = 4     # Gap between blocks at level 0
        STRIDE = BLOCK_WIDTH + GAP_WIDTH
        TREE_RIGHT_MARGIN = 65 # Column index where the tree visualization ends

        # --- DRAW TREE STAGES ---
        # We iterate through stages to build string lines

        # Stage 0: Multipliers
        line_str = ""
        vals = self.pipeline[0]

        # Calculate indent/spacing for Stage 0
        current_stride = STRIDE
        current_indent = 0

        # Build Stage 0 string
        line_str += " " * current_indent
        for i, node in enumerate(vals):
            if node is not None:
                # [A|V] format
                a_val = node['a_val']
                v_val = node['v_val']
                txt = f"[{a_val:>2}|{v_val:>2}]"
            else:
                txt = f"[{'.':>2}|{'.':>2}]"

            line_str += f"{txt:^{BLOCK_WIDTH}}"
            if i < len(vals) - 1:
                line_str += " " * GAP_WIDTH

        lines.append(f"{line_str:<{TREE_RIGHT_MARGIN}}") # Add to output lines

        # Subsequent Adder Stages
        for s in range(1, self.stages + 1):
            prev_stride = current_stride
            current_stride = prev_stride * 2

            # Connector lines (\ /)
            conn_line = ""
            # The center of the previous block is at: indent + (width/2) + index*prev_stride
            # We want slashes connecting parents to children

            # Simplified approach for fixed text layout:
            # Shift indent by half the previous stride roughly
            prev_indent = current_indent
            current_indent = prev_indent + (prev_stride // 2)

            conn_line = " " * current_indent
            data_line = " " * current_indent

            node_count = len(self.pipeline[s])

            for i in range(node_count):
                # Connector
                # A rough visual approx for the tree branches
                conn_txt = "\\   /"
                conn_line += f"{conn_txt:^{BLOCK_WIDTH}}"

                # Data Node
                node = self.pipeline[s][i]
                if node is not None:
                    val = node['val']
                    txt = f"[{val:>3}]"
                else:
                    txt = f"[{'.':>3}]"

                data_line += f"{txt:^{BLOCK_WIDTH}}"

                if i < node_count - 1:
                    # Spacing between nodes at this level
                    # Calculate gap based on stride difference
                    gap_len = current_stride - BLOCK_WIDTH
                    conn_line += " " * gap_len
                    data_line += " " * gap_len

            lines.append(f"{conn_line:<{TREE_RIGHT_MARGIN}}")
            lines.append(f"{data_line:<{TREE_RIGHT_MARGIN}}")

        # --- RIGHT COLUMN: RESULT VECTOR ---
        # We need to append the result vector to the right of these lines.
        # The result vector might be taller than the tree, or shorter.

        final_output = []
        max_visual_rows = max(len(lines), n_rows)

        # Header for the dynamic section
        final_output.append(f"Cycle: {t:<5}")
        final_output.append("-" * 80)

        for r in range(max_visual_rows):
            # Left side: Tree (or empty if below tree)
            left_part = lines[r] if r < len(lines) else " " * TREE_RIGHT_MARGIN

            # Right side: Result Vector element
            # We display the vector building up.
            if r < n_rows:
                if r < len(self.results):
                    res_val = self.results[r]
                    right_part = f"| {res_val:>4} |"
                else:
                    right_part = "|   .  |"
            else:
                right_part = ""

            final_output.append(f"{left_part}{right_part}")

        return "\n".join(final_output)

    def run(self, A, V):
        self.weights = V.flatten()
        n_rows, n_cols = A.shape
        total_cycles = n_rows + self.stages + 2

        expected = np.dot(A, V)

        # Static Header Generation
        static_header = f"""
{'='*80}
ADDER TREE GEMV SIMULATION: {self.size} Inputs
{'='*80}
{self._format_horizontal_matrices(A, V, expected)}

{'='*80}
CYCLE LEVEL SIMULATION
{'='*80}
"""

        self.results = [] # Clear results

        for t in range(total_cycles + 2): # +2 for flush

            # 1. GENERATE DISPLAY (Before state update to show initial state or current latch)
            # However, for consistency with flow, usually we calculate state then show.
            # But to show inputs *entering*, we often setup inputs then show.

            # We will calculate the "Next State" logic but display the "Current State"
            # Actually, let's just render the current pipeline state.

            display_str = self._generate_tree_display(t, n_rows)

            clear_output(wait=True)
            print(static_header)
            print(display_str)

            # 2. UPDATE LOGIC (Flow from Bottom to Top to simulate registers)

            # A. Final Result Collection (Output of last stage)
            last_stage_idx = self.stages
            if self.pipeline[last_stage_idx][0] is not None:
                # We have a valid result from the tree root
                res_node = self.pipeline[last_stage_idx][0]
                # Append to results list
                self.results.append(res_node['val'])

            # B. Adder Stages (Propagate Data)
            # Move from s-1 to s
            for s in range(self.stages, 0, -1):
                input_stage = self.pipeline[s-1]

                # We need pairs of data to produce output
                new_stage_data = []
                for i in range(0, len(input_stage), 2):
                    left = input_stage[i]
                    right = input_stage[i+1]

                    if left is not None and right is not None:
                        # Perform Addition
                        new_sum = left['val'] + right['val']
                        # Inherit row_id (should be same)
                        new_stage_data.append({'val': new_sum, 'row_id': left['row_id']})
                    else:
                        new_stage_data.append(None)

                self.pipeline[s] = new_stage_data

            # C. Multiplier Stage (Input Feed)
            # Load A[row] and V
            if t < n_rows:
                # Stream in new row
                current_row_vals = A[t]
                new_mult_stage = []
                for i in range(self.size):
                    val_a = current_row_vals[i]
                    val_v = self.weights[i]
                    prod = val_a * val_v
                    new_mult_stage.append({
                        'val': prod,
                        'a_val': val_a,
                        'v_val': val_v,
                        'row_id': t
                    })
                self.pipeline[0] = new_mult_stage
            else:
                # No more input, fill with bubbles
                self.pipeline[0] = [None] * self.size

            time.sleep(1.5)


np.random.seed(42)

# Small 4x4
A4 = np.random.randint(1, 5, (4, 4))
B4 = np.random.randint(1, 5, (4, 4))
V4 = np.random.randint(1, 5, (4, 1))

# Large 8x8
A8 = np.random.randint(1, 5, (8, 8))
B8 = np.random.randint(1, 5, (8, 8))
V8 = np.random.randint(1, 5, (8, 1))


### 1. 4x4 GEMM with systolic array

In [2]:
# --- Task 1: 4x4 Systolic Array ---
sim_sys_4 = SystolicArraySimulator(size=4)
sim_sys_4.run(A4, B4)


SYSTOLIC ARRAY GEMM SIMULATION: 4x4
Matrix A               Matrix B               Expected (A@B)      
--------------------------------------------------------------------------------
[[3 4 1 3]             [[4 4 4 3]           = [[36 30 25 30]      
 [3 4 1 1]              [2 1 2 4]           =  [28 22 23 28]      
 [3 2 3 3]              [4 2 2 2]           =  [40 32 25 26]      
 [3 3 4 1]]             [4 4 1 1]]          =  [38 27 27 30]]     

CYCLE LEVEL SIMULATION

Cycle: 11
--------------------------------------------------------------------------------
                         .       .       .       .   
                         .       .       .       .   
                         .       .       .       .   
                         .       .       .       .   
                         ￬       ￬       ￬       ￬   
  .   .   .   .  -> [ 0| 0] [ 0| 0] [ 0| 0] [ 0| 0] 
                     Σ 36   Σ 30   Σ 25   Σ 30  

  .   .   .   .  -> [ 0| 0] [ 0| 0] [ 0| 0] [ 0| 0] 
     

### 2. 8x8 GEMM with systolic array

In [3]:
# --- Task 2: 8x8 Systolic Array ---
sim_sys_8 = SystolicArraySimulator(size=8)
sim_sys_8.run(A8, B8)


SYSTOLIC ARRAY GEMM SIMULATION: 8x8
Matrix A               Matrix B               Expected (A@B)      
--------------------------------------------------------------------------------
[[4 1 1 3 3 3 2 4]     [[3 2 2 4 2 2 2 4]   = [[47 47 53 60 57 45 61 58]
 [4 4 4 3 2 2 3 2]      [2 3 4 3 4 2 3 4]   =  [46 60 71 62 71 51 63 69]
 [3 4 3 4 4 1 3 1]      [1 2 4 1 4 1 2 3]   =  [38 59 68 52 64 50 62 58]
 [3 3 1 1 3 2 4 1]      [1 4 2 1 4 4 4 1]   =  [35 44 54 50 44 39 44 49]
 [4 2 2 2 1 2 1 2]      [1 1 3 1 1 1 3 1]   =  [36 37 44 46 48 34 44 49]
 [4 4 3 4 3 4 1 4]      [4 1 4 4 4 3 3 3]   =  [59 61 76 73 84 57 80 78]
 [3 3 2 1 4 2 4 4]      [1 4 3 3 1 3 1 2]   =  [46 53 64 64 58 44 61 65]
 [2 2 2 2 2 4 2 1]]     [3 2 1 4 3 1 4 4]]  =  [37 38 53 46 51 39 46 46]]

CYCLE LEVEL SIMULATION

Cycle: 23
--------------------------------------------------------------------------------
                                         .       .       .       .       .       .       .       .   
            

### 3. 4x4 GEMV with adder tree

In [4]:
# --- Task 3: 4x4 Adder Tree ---
sim_tree_4 = AdderTreeSimulator(size=4)
sim_tree_4.run(A4, V4)


ADDER TREE GEMV SIMULATION: 4 Inputs
Matrix A                    Vector V          Expected Result
--------------------------------------------------------------------------------
[[3 4 1 3]                * [[4]            = [[25]          
 [3 4 1 1]                *  [2]            =  [23]          
 [3 2 3 3]                *  [2]            =  [25]          
 [3 3 4 1]]               *  [1]]           =  [27]]         

CYCLE LEVEL SIMULATION

Cycle: 9    
--------------------------------------------------------------------------------
 [ .| .]      [ .| .]      [ .| .]      [ .| .]                  |   25 |
        \   /                     \   /                          |   23 |
        [  .]                     [  .]                          |   25 |
                     \   /                                       |   27 |
                     [  .]                                       
